# 06 - Named Entity Recognition: il "cast" e la geografia della narrazione ICE

## Intro

Analisi semantica diacronica del framing NYT sull'agenzia ICE. **Tesi**: il framing è passato da un registro burocratico/istituzionale a una narrazione polarizzata, conflittuale e personalizzata.

Per testarla usiamo la **Named Entity Recognition** (spaCy) su `semantic_text` (headline + abstract + lead) e tracciamo come cambiano, nei quattro periodi storici:
- **PERSON** – chi sono i protagonisti (funzionari/tecnocrati vs politici polarizzanti, attivisti, singoli immigrati);
- **ORG** – quali istituzioni fanno da sfondo (agenzie federali vs tribunali, università, polizie locali, Guardia Nazionale);
- **GPE** – dove si svolge la narrazione (confine/Stati di frontiera vs città santuario e metropoli del Nord).

| Periodo | Etichetta |
|---|---|
| 2003-2016 | The Bureaucratic Era |
| 2017-2020 | The Trump Era |
| 2021-2024 | The Biden Era |
| 2025-Presente | The Current/Conflict Era |

**Limite metodologico**: NER su headline+abstract+lead, non sul full text (paywall NYT, vedi notebook 04 §5). Le entità catturate sono quindi quelle che la redazione mette *in vetrina* – il che, per un'analisi di framing, è un pregio più che un difetto.

## 0. Setup

In [ ]:
import importlib.util
import logging
import re
import subprocess
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import spacy

%matplotlib inline

SPACY_MODEL = "en_core_web_sm"
if not importlib.util.find_spec(SPACY_MODEL):
    print(f"Download modello spaCy '{SPACY_MODEL}'...")
    subprocess.check_call([sys.executable, "-m", "spacy", "download", SPACY_MODEL])

# Serve solo il NER: tagger/parser/lemmatizer disattivati dimezzano i tempi.
# ("tagger"/"parser" potrebbero non esistere in altri modelli, es. en_core_web_trf: spaCy li ignora solo se presenti,
# quindi filtro sui componenti effettivamente nel pipeline.)
_DISABLE = ["tagger", "parser", "attribute_ruler", "lemmatizer"]
_available = spacy.info(SPACY_MODEL)["pipeline"]
nlp = spacy.load(SPACY_MODEL, disable=[c for c in _DISABLE if c in _available])

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = BASE_DIR / "data" / "processed" / "nyt"
PLOTS_DIR = PROCESSED_DIR / "plots"
IN_ICE = PROCESSED_DIR / "articles_ice.parquet"
OUT_ENTITIES = PROCESSED_DIR / "ice_entities.parquet"
OUT_TOP = PROCESSED_DIR / "ice_ner_top_by_period.csv"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)
log = logging.getLogger("ner_analysis")

ENTITY_LABELS = ["PERSON", "ORG", "GPE"]
TOP_N = 10
# Un'entità citata in un solo articolo è aneddoto, non "cast": nel periodo 2021-2024 (96 articoli) la coda del top 10
# era fatta di singoletti, spesso errori NER su headline in Title Case. Un top con meno di 10 voci è preferibile.
MIN_ARTICLES = 2

print(f"spaCy pipeline attiva: {nlp.pipe_names}")
print(f"BASE_DIR: {BASE_DIR}")

## 1. Caricamento e segmentazione temporale

`semantic_text` è ricostruito come nel notebook 04 (headline + abstract + lead). In più rimuoviamo il **dateline** iniziale del lead (es. `WASHINGTON —`, `LOS ANGELES —`): indica la città da cui il giornalista ha *firmato* il pezzo, non la geografia della storia. Lasciarlo gonfierebbe artificialmente Washington (~100 articoli) e Los Angeles nei GPE.

In [ ]:
PERIODS = [
    ("2003-2016", "The Bureaucratic Era", 2003, 2016),
    ("2017-2020", "The Trump Era", 2017, 2020),
    ("2021-2024", "The Biden Era", 2021, 2024),
    ("2025-Presente", "The Current/Conflict Era", 2025, 9999),
]
PERIOD_ORDER = [p[0] for p in PERIODS]
PERIOD_TITLES = {p[0]: f"{p[0]}\n{p[1]}" for p in PERIODS}

# Dateline NYT: "CITTÀ[, STATO] —" in maiuscolo a inizio lead
DATELINE_RE = re.compile(r"^[A-Z][A-Z .,'’-]{2,40}?\s*[—–]\s*")


def assign_period(year: int) -> str | None:
    for label, _, start, end in PERIODS:
        if start <= year <= end:
            return label
    return None


log.info(f"Caricamento {IN_ICE.name}...")
df = pd.read_parquet(IN_ICE)

for col in ["headline", "abstract", "lead_paragraph"]:
    df[col] = df[col].fillna("").astype(str)

df["lead_clean"] = df["lead_paragraph"].str.replace(DATELINE_RE, "", n=1, regex=True)
# abstract e lead spesso coincidono: evitiamo di duplicare il testo (e quindi le entità)
same = df["abstract"].str.strip() == df["lead_paragraph"].str.strip()
df.loc[same, "lead_clean"] = ""

df["semantic_text"] = (
    (df["headline"] + ". " + df["abstract"] + " " + df["lead_clean"])
    .str.replace(r"\s+", " ", regex=True)
    .str.strip(" .")
)

df["pub_date"] = pd.to_datetime(df["pub_date"], utc=True, errors="coerce")
n_bad_date = df["pub_date"].isna().sum()
if n_bad_date:
    log.warning(f"{n_bad_date} articoli con pub_date non valida: esclusi")
df = df.dropna(subset=["pub_date"])
df["period"] = pd.Categorical(df["pub_date"].dt.year.map(assign_period), categories=PERIOD_ORDER, ordered=True)

empty = df["semantic_text"].str.len() == 0
if empty.any():
    log.warning(f"{empty.sum()} articoli con semantic_text vuoto: esclusi")
df = df[~empty].reset_index(drop=True)

period_sizes = df["period"].value_counts().reindex(PERIOD_ORDER)
log.info(f"Articoli per periodo:\n{period_sizes.to_string()}")
print(f"Righe con dateline rimosso: {(df['lead_clean'] != df['lead_paragraph']).sum() - same.sum()}")

## 2. Estrazione delle entità

`nlp.pipe()` in batch su tutti i testi (~2k articoli: pochi secondi con `en_core_web_sm`). Il modello small è una scelta consapevole: per entità "di vetrina" in prosa giornalistica pulita la precisione è sufficiente e l'analisi resta rieseguibile su CPU in pochi secondi; per una verifica di robustezza basta cambiare `SPACY_MODEL` in `en_core_web_trf`.

L'output grezzo del NER è rumoroso e **non** va contato così com'è. Problemi reali osservati su questo corpus:

| Problema | Esempio | Trattamento |
|---|---|---|
| Possessivi | `Trump’s`, `Donald J. Trump’s` | strip `'s`/`’s` |
| Articolo iniziale | `the Department of Homeland Security` | strip `the` |
| Iniziali intermedie | `Donald J. Trump`, `Thomas D. Homan` | rimosse → `Donald Trump` |
| Solo cognome | `Trump`, `Obama`, `Biden` | ricondotto al nome completo più frequente con quel cognome |
| Varianti della stessa entità | `U.S.`, `the United States`, `America` | alias map manuale |
| Abbreviazioni AP degli Stati | `Ore.`, `Ariz.`, `Calif.` | mappate al nome esteso |
| Errori di label | `Trump` come ORG, `Border Patrol` come PERSON | relabel esplicito |
| Boilerplate podcast | `Via Stitcher`, `Via RadioPublic` | scartati |

**Unità di conteggio = articolo** (document frequency): un'entità conta una volta per articolo anche se ripetuta in titolo, abstract e lead. Evita che un singolo pezzo con 4 menzioni pesi quanto 4 articoli.

In [ ]:
texts = df["semantic_text"].tolist()
raw_ents: list[list[tuple[str, str]]] = []

log.info(f"NER su {len(texts)} testi con {SPACY_MODEL}...")
for doc in nlp.pipe(texts, batch_size=256):
    raw_ents.append([(ent.text, ent.label_) for ent in doc.ents if ent.label_ in ENTITY_LABELS])

n_raw = sum(len(e) for e in raw_ents)
log.info(f"Entità grezze PERSON/ORG/GPE estratte: {n_raw:,}")

### 2.1 Normalizzazione

Tutte le mappe sono esplicite e piccole, così ogni scelta resta ispezionabile e citabile nel report. `EXCLUDE` rimuove le entità **tautologiche**: ICE stessa (presente per costruzione in ogni articolo, come già escluso nel topic network del notebook 02), il NYT che cita sé stesso e gli Stati Uniti come Paese (sfondo universale che schiaccerebbe ogni altro GPE: "dove" ci interessa a livello sub-nazionale ed estero).

In [ ]:
STATE_ABBR = {
    "Ala.": "Alabama", "Ariz.": "Arizona", "Ark.": "Arkansas", "Calif.": "California", "Colo.": "Colorado",
    "Conn.": "Connecticut", "Del.": "Delaware", "Fla.": "Florida", "Ga.": "Georgia", "Ill.": "Illinois",
    "Ind.": "Indiana", "Kan.": "Kansas", "Ky.": "Kentucky", "La.": "Louisiana", "Md.": "Maryland",
    "Mass.": "Massachusetts", "Mich.": "Michigan", "Minn.": "Minnesota", "Miss.": "Mississippi", "Mo.": "Missouri",
    "Mont.": "Montana", "Neb.": "Nebraska", "Nev.": "Nevada", "N.H.": "New Hampshire", "N.J.": "New Jersey",
    "N.M.": "New Mexico", "N.Y.": "New York", "N.C.": "North Carolina", "N.D.": "North Dakota", "Okla.": "Oklahoma",
    "Ore.": "Oregon", "Pa.": "Pennsylvania", "R.I.": "Rhode Island", "S.C.": "South Carolina", "S.D.": "South Dakota",
    "Tenn.": "Tennessee", "Va.": "Virginia", "Vt.": "Vermont", "Wash.": "Washington State", "Wis.": "Wisconsin",
    "W.Va.": "West Virginia", "Wyo.": "Wyoming",
}

ALIASES = {
    # GPE
    "U.S.": "United States", "U.S": "United States", "US": "United States", "America": "United States", "the U.S.": "United States",
    "U.S.A.": "United States", "N.Y.C.": "New York City", "L.A.": "Los Angeles", "D.C.": "Washington, D.C.",
    # ORG
    "Immigration and Customs Enforcement": "ICE", "Customs Enforcement": "ICE", "I.C.E.": "ICE",
    "U.S. Immigration and Customs Enforcement": "ICE", "Immigration": "ICE", "Customs": "ICE",
    "U.S. Immigration": "ICE", "Immigration and": "ICE",  # span troncati di "Immigration and Customs Enforcement"
    "Department of Homeland Security": "Department of Homeland Security", "Homeland Security": "Department of Homeland Security",
    "Homeland Security Department": "Department of Homeland Security", "D.H.S.": "Department of Homeland Security",
    "DHS": "Department of Homeland Security",
    "Justice Department": "Department of Justice", "Department of Justice": "Department of Justice",
    "Customs and Border Protection": "Customs and Border Protection", "C.B.P.": "Customs and Border Protection",
    "G.O.P.": "Republican Party", "Republicans": "Republican Party", "Democrats": "Democratic Party",
    "New York Times": "The New York Times", "Times": "The New York Times",
    "F.B.I.": "FBI", "N.Y.P.D.": "NYPD", "A.C.L.U.": "ACLU", "American Civil Liberties Union": "ACLU",
    "Trump Administration": "Trump administration", "Biden Administration": "Biden administration",
    "Obama Administration": "Obama administration",
    "A.C.L.U.": "ACLU", "Columbia University": "Columbia", "Harvard University": "Harvard",
    # PERSON
    "Anna Sorokin": "Anna Sorokin (Delvey)", "Anna Delvey": "Anna Sorokin (Delvey)",  # stessa persona, nome d'arte
    "Sorokin": "Anna Sorokin (Delvey)", "Delvey": "Anna Sorokin (Delvey)",
    "Tom Homan": "Thomas Homan", "Abrego Garcia": "Kilmar Armando Abrego Garcia",
    "Kilmar Abrego Garcia": "Kilmar Armando Abrego Garcia",
}

# (testo normalizzato, label spaCy) -> label corretta, None = scarta
RELABEL = {
    ("Trump", "ORG"): "PERSON",
    ("Biden", "ORG"): "PERSON",
    ("Obama", "ORG"): "PERSON",
    ("Kirstjen Nielsen", "ORG"): "PERSON",
    ("ACLU", "PERSON"): "ORG",
    ("Twitter", "PERSON"): "ORG",
    ("Android", "PERSON"): None,
    ("Android", "ORG"): None,
    ("U.S.-Mexico", "ORG"): None,
    ("Department of Homeland Security", "GPE"): "ORG",
    ("Delaney Hall", "PERSON"): None,   # centro di detenzione a Newark (FAC), non una persona       # aggettivo ("U.S.-Mexico border"), non un'organizzazione
    ("Border Patrol", "PERSON"): "ORG",
    ("Deportations", "PERSON"): None,
    ("Guatemalan", "ORG"): None,        # NORP etichettato ORG
    ("A.I.", "ORG"): None,
    ("Social Security", "ORG"): None,   # quasi sempre "Social Security number", non l'agenzia
    ("Washington", "ORG"): None,        # dateline sfuggiti al regex / metonimia
}

NOISE = {"Via Stitcher", "Via RadioPublic", "Via Apple Podcasts", "Via Spotify", "Via Google Podcasts",
         # headline in Title Case scambiate per nomi propri
         "Broken", "Accused of", "Adams Meets Trump", "Rules", "Trump’s Immigration Crackdown",
         # ORG generiche (nomi comuni capitalizzati), non istituzioni identificabili
         "Immigrants", "Immigration Agents", "Latino", "Border", "State"}
# Headline NYT in Title Case vengono a volte etichettate per intero come ORG/PERSON
# ("Biden Is Hoping Small Changes Go a Long Way on Immigration"): nessun nome proprio reale supera 5 token
MAX_ENTITY_TOKENS = 5
# Stesso problema sotto i 5 token: "Biden Shut the Border", "Biden Immigration Policy", "Trump’s Immigration Crackdown".
# Un nome presidenziale seguito da altro testo non è mai un'entità propria, salvo "<X> administration" (gestito dagli alias).
HEADLINE_FRAGMENT_RE = re.compile(r"^(?:Trump|Biden|Obama|Bush)(?:[’']s)?\s+(?!administration$)", flags=re.IGNORECASE)

EXCLUDE = {
    "ORG": {"ICE", "The New York Times"},
    "GPE": {"United States"},
    "PERSON": set(),
}

POSSESSIVE_RE = re.compile(r"[’']s?$")
LEADING_THE_RE = re.compile(r"^the\s+", flags=re.IGNORECASE)
MIDDLE_INITIAL_RE = re.compile(r"(?<=\w)\s+[A-Z]\.(?=\s+\w)")


def normalize(text: str, label: str) -> tuple[str, str] | None:
    t = POSSESSIVE_RE.sub("", text.strip())
    # il "." finale NON va tolto: fa parte di U.S., G.O.P., A.C.L.U., Ore. (e degli alias sotto)
    t = LEADING_THE_RE.sub("", t).strip(" ,;:\"“”")
    if not t or t in NOISE or len(t) < 2 or len(t.split()) > MAX_ENTITY_TOKENS:
        return None
    # dateline/titoli in maiuscolo sopravvissuti ("LOS ANGELES"); acronimi corti o puntati restano intatti
    if t.isupper() and len(t) > 4 and "." not in t:
        t = t.title()
    t = STATE_ABBR.get(t, t) if label == "GPE" else t
    t = ALIASES.get(t, t)
    if label == "PERSON":
        t = MIDDLE_INITIAL_RE.sub("", t)
    if label in ("PERSON", "ORG") and HEADLINE_FRAGMENT_RE.match(t) and not t.lower().endswith("administration"):
        return None
    if (t, label) in RELABEL:
        label = RELABEL[(t, label)]
        if label is None:
            return None
    return t, label


norm_ents = [[n for e in doc_ents if (n := normalize(*e)) is not None] for doc_ents in raw_ents]

### 2.2 Cognome → nome completo

Il NYT usa il nome completo alla prima menzione e poi solo il cognome ("Mr. Trump"). spaCy restituisce quindi `Trump` e `Donald Trump` come entità distinte. Ricostruiamo la mappa **a livello di corpus**: un cognome isolato viene ricondotto al nome completo più frequente che termina con quel cognome, *solo se* quel nome copre almeno il 60% delle varianti (soglia contro gli omonimi: se "Johnson" corrisponde a più persone diverse con frequenza simile, resta non risolto).

In [ ]:
SURNAME_DOMINANCE = 0.6

full_names = Counter(t for doc in norm_ents for t, l in doc if l == "PERSON" and " " in t)
by_surname: dict[str, Counter] = {}
for name, c in full_names.items():
    by_surname.setdefault(name.split()[-1], Counter())[name] = c

SURNAME_MAP = {}
for surname, variants in by_surname.items():
    name, c = variants.most_common(1)[0]
    if c / sum(variants.values()) >= SURNAME_DOMINANCE:
        SURNAME_MAP[surname] = name

records = []
for idx, doc in enumerate(norm_ents):
    seen = set()
    for t, l in doc:
        if l == "PERSON" and " " not in t:
            t = SURNAME_MAP.get(t, t)
        if t in EXCLUDE[l] or (t, l) in seen:
            continue
        seen.add((t, l))
        records.append((idx, t, l))

ents = pd.DataFrame(records, columns=["row", "entity", "label"])
ents = ents.join(df[["id", "pub_date", "period"]], on="row").drop(columns="row")

ents.to_parquet(OUT_ENTITIES, index=False)
log.info(f"Coppie articolo-entità (dedup per articolo): {len(ents):,} -> {OUT_ENTITIES.name}")

check = ["Trump", "Obama", "Biden", "Homan", "Noem", "Napolitano", "Adams"]
print("Esempi di risoluzione cognome -> nome completo:")
for s in check:
    print(f"  {s:12s} -> {SURNAME_MAP.get(s, '(non risolto)')}")

## 3. Top 10 entità per periodo e categoria

Solo entità presenti in almeno `MIN_ARTICLES = 2` articoli del periodo. Oltre al conteggio assoluto calcoliamo la **quota di articoli del periodo** che menziona l'entità. È la metrica da confrontare tra periodi: il corpus 2025-Presente è ~5 volte più grande del 2003-2016, quindi i conteggi assoluti premierebbero meccanicamente il periodo recente.

In [ ]:
counts = (
    ents.groupby(["period", "label", "entity"], observed=True)
    .size()
    .rename("n_articles")
    .reset_index()
)
counts["share"] = counts["n_articles"] / counts["period"].map(period_sizes).astype(float)

top = (
    counts[counts["n_articles"] >= MIN_ARTICLES]
    .sort_values(["period", "label", "n_articles", "entity"], ascending=[True, True, False, True])
    .groupby(["period", "label"], observed=True)
    .head(TOP_N)
    .reset_index(drop=True)
)
top["rank"] = top.groupby(["period", "label"], observed=True).cumcount() + 1
top.to_csv(OUT_TOP, index=False)
log.info(f"Top-{TOP_N} salvato in {OUT_TOP.name}")

for label in ENTITY_LABELS:
    wide = (
        top[top["label"] == label]
        .assign(cell=lambda d: d["entity"] + " (" + (d["share"] * 100).round(1).astype(str) + "%)")
        .pivot(index="rank", columns="period", values="cell")
        .reindex(columns=PERIOD_ORDER)
    )
    print(f"\n=== {label} — top {TOP_N} (% articoli del periodo) ===")
    display(wide)

## 4. Visualizzazione

Griglia 3×4: righe = categoria (PERSON / ORG / GPE), colonne = periodo. Asse x = % di articoli del periodo che citano l'entità, **condiviso lungo ogni riga** così la lunghezza delle barre è confrontabile tra ere (una barra più lunga nel 2025 significa davvero maggiore centralità, non solo più articoli). Un colore per categoria; il nome della categoria è comunque scritto sull'asse, quindi il colore non è l'unico veicolo d'identità.

In [ ]:
LABEL_COLORS = {"PERSON": "#2a78d6", "ORG": "#eb6834", "GPE": "#1baf7a"}
LABEL_NAMES = {"PERSON": "Persone (PERSON)", "ORG": "Organizzazioni (ORG)", "GPE": "Luoghi (GPE)"}


def plot_ner_grid(top: pd.DataFrame, period_sizes: pd.Series, output_path: Path | None = None) -> plt.Figure:
    fig, axes = plt.subplots(
        nrows=len(ENTITY_LABELS), ncols=len(PERIOD_ORDER),
        figsize=(24, 17), gridspec_kw={"hspace": 0.32, "wspace": 0.95},
    )
    for r, label in enumerate(ENTITY_LABELS):
        row_max = top.loc[top["label"] == label, "share"].max() * 100
        for c, period in enumerate(PERIOD_ORDER):
            ax = axes[r, c]
            sub = top[(top["label"] == label) & (top["period"] == period)].sort_values("rank", ascending=False)
            # slot fissi dall'alto: con meno di TOP_N voci le barre non si ingrossano e il rank resta allineato tra colonne
            y = [TOP_N - 1 - i for i in range(len(sub))][::-1]
            ax.barh(y, sub["share"] * 100, height=0.62, color=LABEL_COLORS[label], edgecolor="white", linewidth=2)
            ax.set_yticks(list(y), sub["entity"], fontsize=10.5)
            ax.set_ylim(-0.6, TOP_N - 0.4)
            for yi, share, n in zip(y, sub["share"] * 100, sub["n_articles"]):
                ax.text(share + row_max * 0.02, yi, f"{share:.1f}% ({n})", va="center", fontsize=8.5, color="#555555")
            ax.set_xlim(0, row_max * 1.45)
            ax.xaxis.grid(True, color="#e6e6e6", linewidth=0.8)
            ax.yaxis.grid(False)
            ax.set_axisbelow(True)
            for side in ["top", "right", "left"]:
                ax.spines[side].set_visible(False)
            ax.spines["bottom"].set_color("#bbbbbb")
            ax.tick_params(axis="x", labelsize=9, colors="#555555")
            ax.tick_params(axis="y", length=0)
            if r == 0:
                ax.set_title(f"{PERIOD_TITLES[period]}\n(n = {period_sizes[period]:,} articoli)",
                             fontsize=12.5, fontweight="bold", pad=12)
            if c == 0:
                ax.set_ylabel(LABEL_NAMES[label], fontsize=13, fontweight="bold", labelpad=14)
            if r == len(ENTITY_LABELS) - 1:
                ax.set_xlabel("% articoli del periodo", fontsize=10, color="#555555")
    fig.suptitle(
        "Il \"cast\" e la geografia della narrazione NYT su ICE, 2003-2026\n"
        "Top 10 entità per categoria e periodo — quota di articoli che le menzionano (headline + abstract + lead)",
        fontsize=16, y=0.995,
    )
    fig.text(0.5, 0.005,
             "Esclusi: ICE, The New York Times, United States (entità tautologiche). "
             f"Conteggio = articoli distinti. NER: spaCy {SPACY_MODEL}.",
             ha="center", fontsize=9.5, color="#777777")
    if output_path:
        fig.savefig(output_path, dpi=200, bbox_inches="tight")
        log.info(f"Grafico salvato in {output_path}")
    return fig


fig = plot_ner_grid(top, period_sizes, PLOTS_DIR / "ner_top_entities_by_period.png")
plt.show()

### 4.1 Traiettorie delle entità chiave

La griglia mostra *chi entra* nel top 10; questa vista mostra *come varia il peso* di un insieme fisso di entità "diagnostiche" per la tesi. Le entità sono scelte fra quelle presenti nei top 10 sopra, divise per ruolo sociologico.

In [ ]:
def plot_entity_trajectories(counts: pd.DataFrame, groups: dict[str, list[tuple[str, str]]], output_path: Path | None = None) -> plt.Figure:
    fig, axes = plt.subplots(1, len(groups), figsize=(8 * len(groups), 5), sharey=False)
    palette = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
    for ax, (title, members) in zip(axes, groups.items()):
        for i, (entity, label) in enumerate(members):
            s = (
                counts[(counts["entity"] == entity) & (counts["label"] == label)]
                .set_index("period")["share"].reindex(PERIOD_ORDER).fillna(0) * 100
            )
            ax.plot(PERIOD_ORDER, s.values, marker="o", markersize=8, linewidth=2, color=palette[i], label=entity)
        ax.set_title(title, fontsize=12, fontweight="bold")
        ax.set_ylabel("% articoli del periodo")
        ax.tick_params(axis="x", labelsize=9)
        ax.legend(fontsize=9, frameon=False, loc="upper left", bbox_to_anchor=(1.0, 1.0))
        ax.yaxis.grid(True, color="#e6e6e6", linewidth=0.8)
        ax.xaxis.grid(False)
        for side in ["top", "right"]:
            ax.spines[side].set_visible(False)
    fig.tight_layout()
    if output_path:
        fig.savefig(output_path, dpi=200, bbox_inches="tight")
        log.info(f"Grafico salvato in {output_path}")
    return fig


# Ogni entità è verificata sul corpus: quelle assenti vengono segnalate invece di disegnare una linea piatta a 0 fuorviante
TRAJECTORY_GROUPS = {
    "Istituzioni federali vs contesto conflittuale": [
        ("Department of Homeland Security", "ORG"), ("Congress", "ORG"),
        ("Supreme Court", "ORG"), ("National Guard", "ORG"),
    ],
    "Geografia: confine vs città US": [
        ("Texas", "GPE"), ("Arizona", "GPE"), ("Mexico", "GPE"),
        ("Chicago", "GPE"), ("Minneapolis", "GPE"), ("New York City", "GPE"),
    ],
}
known = set(zip(counts["entity"], counts["label"]))
for g, members in TRAJECTORY_GROUPS.items():
    missing = [m for m in members if m not in known]
    if missing:
        log.warning(f"[{g}] entità assenti dal corpus, rimosse: {missing}")
    TRAJECTORY_GROUPS[g] = [m for m in members if m in known]

fig = plot_entity_trajectories(counts, TRAJECTORY_GROUPS, PLOTS_DIR / "ner_entity_trajectories.png")
plt.show()

## 5. Indicatori di personalizzazione

La tesi della *personalizzazione* ha una controparte quantitativa diretta, indipendente dai singoli nomi:
- **% articoli con almeno una PERSON** – quanto spesso la storia ha un volto;
- **PERSON distinte per 100 articoli** – quanto è ampio il cast (molti individui diversi = storie di singoli immigrati/attivisti, non solo pochi vertici istituzionali);
- **quota top-3 PERSON** – concentrazione: se pochi nomi monopolizzano il discorso (es. il Presidente), la personalizzazione è *verticale* (leader polarizzante); se il cast è ampio e diffuso, è *orizzontale* (vittime, attivisti, casi individuali).
- **rapporto ORG/PERSON** – un valore in calo indica il passaggio da narrazione istituzionale a narrazione centrata sulle persone.

In [ ]:
def personalization_metrics(ents: pd.DataFrame, period_sizes: pd.Series) -> pd.DataFrame:
    rows = []
    for period in PERIOD_ORDER:
        e = ents[ents["period"] == period]
        n = period_sizes[period]
        pers = e[e["label"] == "PERSON"]
        pers_counts = pers["entity"].value_counts()
        rows.append({
            "period": period,
            "articoli": n,
            "% articoli con PERSON": 100 * pers["id"].nunique() / n,
            "PERSON distinte / 100 art.": 100 * pers_counts.size / n,
            "quota top-3 PERSON (%)": 100 * pers_counts.head(3).sum() / max(pers_counts.sum(), 1),
            "menzioni ORG / PERSON": (e["label"] == "ORG").sum() / max(len(pers), 1),
            "GPE distinti / 100 art.": 100 * e.loc[e["label"] == "GPE", "entity"].nunique() / n,
        })
    return pd.DataFrame(rows).set_index("period").round(2)


metrics = personalization_metrics(ents, period_sizes)
display(metrics)

## 6. Chiave di lettura sociologica

Cosa cercare nell'output per confermare o falsificare la tesi *burocratico → polarizzato/conflittuale/personalizzato*. Le ipotesi sono formulate **prima** di guardare i dati e vanno confrontate onestamente con quanto sopra: se un pattern atteso non compare, è un risultato, non un errore da correggere.

**PERSON — dal funzionario al leader polarizzante e al singolo individuo**
- *2003-2016 (atteso)*: tecnocrati e vertici amministrativi (Segretari DHS come Napolitano/Chertoff, direttori ICE), presidenti citati in veste istituzionale (Bush, Obama per le politiche di rimpatrio), sceriffi come Arpaio come eccezione "locale". Personalizzazione bassa, cast ristretto.
- *2017-2020*: centralizzazione su **Trump** (quota top-3 alta = personalizzazione verticale), ingresso di figure politiche polarizzanti (Sessions, Kelly, Homan) e dei primi **casi individuali** (es. Ravi Ragbir – attivista minacciato di deportazione).
- *2021-2024*: se la tesi regge, il cast torna parzialmente istituzionale (Biden, Mayorkas) e **politici statali antagonisti** (Abbott – bus di migranti verso le città del Nord) spostano il conflitto dall'asse federale all'asse Stato-vs-Stato.
- *2025-Presente*: attesa la co-presenza di vertici esecutivi (Trump, Noem, Homan), **singoli detenuti/deportati** diventati casi mediatici (Abrego Garcia, Mahmoud Khalil, Mohsen Mahdawi) e politici locali in opposizione. Un cast ampio (molte PERSON distinte / 100 art.) segnala la personalizzazione *orizzontale*: la storia non è più "l'agenzia" ma "le persone colpite".

**ORG — dallo sfondo amministrativo all'arena del conflitto**
- Declino relativo di DHS/Congress come frame dominante = meno lettura "policy/burocrazia".
- Crescita di **tribunali** (Supreme Court, corti federali) = giudiziarizzazione del conflitto.
- Comparsa di **università** (Columbia, Harvard) = estensione della repressione a spazi accademici/libertà di parola (casi Khalil/Öztürk).
- Comparsa di **National Guard**, polizie locali, City Council = militarizzazione e scontro federale-locale.
- Partiti (Republican/Democratic Party) nel top 10 = lettura esplicitamente partitica/polarizzata.

**GPE — dal confine alle città santuario**
- *Fase burocratica*: geografia di **frontiera** (Texas, Arizona, Mexico) e dei **Paesi d'origine** (Guatemala, Honduras, El Salvador) – ICE come apparato di controllo dei flussi.
- *Fase di conflitto*: spostamento verso **metropoli dell'interno e città santuario** (Chicago, Minneapolis, Portland, Los Angeles, New York City, Newark) – ICE come presenza nelle comunità, raid urbani, proteste. Un GPE come Minnesota/Minneapolis o Portland in top 10 è un indicatore diretto di eventi di scontro in piazza, non di politica migratoria in senso stretto.
- Stati "blu" (California, Illinois, New Jersey) in crescita rispetto a quelli di confine = conflitto federale-statale lungo linee partitiche.

**Cautele interpretative**
- NER ≠ framing: un'entità citata non dice *come* è rappresentata. Questo notebook descrive il **cast**; il *tono* verso quel cast va misurato con il sentiment (step successivo in checklist).
- Il periodo 2025-Presente ha byline vuote e parte degli articoli live-update non taggati (vedi notebook 03 e checklist): il corpus di quell'era è sbilanciato verso pezzi brevi/breaking, che per natura sono più "event-driven".
- Il periodo 2021-2024 conta solo **96 articoli**: nel suo top 10 un'entità a 2-3 articoli pesa già 2-3%, quindi le posizioni oltre la 4ª-5ª sono statisticamente fragili (singoli casi di cronaca, es. Anna Sorokin).
- `en_core_web_sm` confonde a volte PERSON/ORG/GPE (es. `Washington` persona/luogo/istituzione); le mappe di §2.1 correggono i casi frequenti, i residui nella coda lunga non spostano i top 10.